**1. Upload file ZIP**

In [ ]:
from google.colab import files
uploaded = files.upload()

TypeError: 'NoneType' object is not subscriptable

**2. Extract Datase**

In [ ]:
import zipfile

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

data_dir = '/content/dataset'

**3. Cek Struktur**

In [ ]:
import os

data_dir = '/content/dataset/dentalscan'
print(os.listdir(data_dir))

**4. Import Library**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

from sklearn.metrics import classification_report, confusion_matrix

**5. Load Data**

In [ ]:
data_dir = '/content/dataset/dentalscan'  # sesuaikan path kamu

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    data_dir,
    target_size=(224,224),
    batch_size=8,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    data_dir,
    target_size=(224,224),
    batch_size=8,
    class_mode='binary',
    subset='validation'
)

**6. Labeling Dan Cleaning**

In [ ]:
clean_images = []
clean_labels = []

for images, labels in train_data:
    for i in range(len(images)):
        img = images[i]
        label = labels[i]

        # Cleaning
        if img is None:
            continue

        if img.shape != (224, 224, 3):
            continue

        # Filter brightness
        if np.mean(img) < 0.05 or np.mean(img) > 0.95:
            continue

        clean_images.append(img)
        clean_labels.append(label)

    # stop biar nggak infinite loop
    if len(clean_images) >= 100:
        break

clean_images = np.array(clean_images)
clean_labels = np.array(clean_labels)

print("Jumlah data setelah cleaning:", len(clean_images))

**7. TAMPILKAN GAMBAR + LABEL**

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img, img_to_array

# cek isi folder
print(os.listdir('/content/dataset/dentalscan'))
print(os.listdir('/content/dataset/dentalscan/dentalscan'))

# path folder class
caries_path = "/content/dataset/dentalscan/dentalscan/gigi caries"
normal_path = "/content/dataset/dentalscan/dentalscan/gigi sehat"
gusi_path = "/content/dataset/dentalscan/dentalscan/gusi sehat"
karang_path = "/content/dataset/dentalscan/dentalscan/karang gigi"

# ambil masing-masing 3 gambar random
caries_images = random.sample(os.listdir(caries_path), 3)
normal_images = random.sample(os.listdir(normal_path), 3)
gusi_images = random.sample(os.listdir(gusi_path), 3)
karang_images = random.sample(os.listdir(karang_path), 3)

# gabungkan semua gambar + label
all_images = (
    [(img, "Gigi Caries", caries_path) for img in caries_images] +
    [(img, "Gigi Sehat", normal_path) for img in normal_images] +
    [(img, "Gusi Sehat", gusi_path) for img in gusi_images] +
    [(img, "Karang Gigi", karang_path) for img in karang_images]
)

# tampilkan gambar
plt.figure(figsize=(15,10))

for i, (img_name, label, folder) in enumerate(all_images):
    img_path = os.path.join(folder, img_name)

    img = load_img(img_path, target_size=(150,150))
    img = img_to_array(img).astype("uint8")

    plt.subplot(4,3,i+1)
    plt.imshow(img)
    plt.title(label)
    plt.axis('off')

plt.tight_layout()
plt.show()

**8. Image Augmentation**

In [ ]:
# ==============================
# 1. IMPORT
# ==============================
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ==============================
# 2. FUNGSI AUGMENTASI
# ==============================
def augment_image(image):
    augmented = []
    h, w = image.shape[:2]

    # Rotasi
    for angle in [-20, 20]:
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
        rotated = cv2.warpAffine(image, M, (w, h))
        augmented.append(rotated)

    # Zoom
    zoom = image[int(0.1*h):int(0.9*h), int(0.1*w):int(0.9*w)]
    zoom = cv2.resize(zoom, (w, h))
    augmented.append(zoom)

    # Brightness
    bright = cv2.convertScaleAbs(image, alpha=1, beta=40)
    augmented.append(bright)

    return augmented

# ==============================
# 3. AMBIL DATA DARI GENERATOR
# ==============================
images, labels = next(train_data)

# ==============================
# 4. AUGMENTASI
# ==============================
sample_img = (images[0] * 255).astype(np.uint8)
aug_images = augment_image(sample_img)

# ==============================
# 5. VISUALISASI (TANPA UBAH WARNA)
# ==============================
plt.figure(figsize=(12,4))

# original
plt.subplot(1, len(aug_images)+1, 1)
plt.imshow(sample_img)
plt.title("Original")
plt.axis('off')

# augmentasi
for i, img in enumerate(aug_images):
    plt.subplot(1, len(aug_images)+1, i+2)
    plt.imshow(img)
    plt.title(f"Aug {i+1}")
    plt.axis('off')

plt.show()

**9. Exploratory Data Analysis (EDA)**

In [ ]:
# ====================================
# EDA DATASET DENTALSCAN
# ====================================

import os
import pandas as pd
from PIL import Image

# path dataset
dataset_path = "/content/dataset/dentalscan/dentalscan"

# ambil nama folder kelas
classes = os.listdir(dataset_path)

# list untuk menyimpan data
data = []

# membaca gambar pada dataset
for label in classes:

    folder_path = os.path.join(dataset_path, label)

    if os.path.isdir(folder_path):

        for img_name in os.listdir(folder_path):

            img_path = os.path.join(folder_path, img_name)

            try:
                # membuka gambar
                img = Image.open(img_path)

                # ukuran gambar
                width, height = img.size

                # simpan data
                data.append({
                    "Label": label,
                    "Width": width,
                    "Height": height,
                    "Filename": img_name
                })

            except:
                print(f"Gagal membaca gambar: {img_name}")

# membuat dataframe
df = pd.DataFrame(data)

# ====================================
# MENAMPILKAN INFORMASI DATASET
# ====================================

print("===== INFORMASI DATASET =====")
print(f"Jumlah Data : {len(df)}")
print(f"Jumlah Kolom : {len(df.columns)}")

print("\n===== 5 DATA PERTAMA =====")
print(df.head())

print("\n===== INFORMASI DATA =====")
df.info()

# ====================================
# JUMLAH DATA TIAP KELAS
# ====================================

print("\n===== JUMLAH DATA TIAP KELAS =====")
print(df["Label"].value_counts())

# ====================================
# STATISTIK UKURAN GAMBAR
# ====================================

print("\n===== STATISTIK LEBAR GAMBAR =====")
print(df["Width"].describe())

print("\n===== STATISTIK TINGGI GAMBAR =====")
print(df["Height"].describe())

# ====================================
# CEK MISSING VALUE
# ====================================

print("\n===== MISSING VALUE =====")
print(df.isnull().sum())

# ====================================
# CEK DATA DUPLIKAT
# ====================================

print("\n===== DATA DUPLIKAT =====")
print(df.duplicated().sum())